# Filesystem: thread state and shared store

`FilesystemMiddleware` adds virtual file tools. `StateBackend` keeps `/notes/decision.txt` in one graph thread. A separate agent routes `/memories/preference.txt` through `StoreBackend`, sharing it between writer and reader threads. Paths outside that route remain local to their thread.

`InMemorySaver` and `InMemoryStore` retain data only for the lifetime of their objects. Cross-thread sharing and persistence across process restarts are different properties; durable storage requires a durable backend.

[Filesystem backends](https://docs.langchain.com/oss/python/deepagents/backends)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from lab.custom import demo_namespace
from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from model_config import get_model, run_config
from lab.faults import ScriptedModel, tool_request
from langchain_core.messages import AIMessage


In [ ]:
# Execute
state_agent = create_agent(
    model=get_model(),
    middleware=[FilesystemMiddleware(backend=StateBackend()), TraceMiddleware("11_state_files", pretty=True)],
    checkpointer=InMemorySaver(),
)
state_config = run_config("file-thread")
state_written = state_agent.invoke({"messages": [("user", prompt("filesystem_write"))]}, config=state_config)
virtual_files = state_agent.get_state(state_config).values.get("files")

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(state_written["messages"], pretty=True)
print("Virtual files:", virtual_files)

In [ ]:
# Execute: this remains the state-backed agent even after the store example.
state_read_start = len(state_agent.get_state(state_config).values["messages"])
state_read = state_agent.invoke({"messages": [("user", prompt("filesystem_read"))]}, config=state_config)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(state_read["messages"][state_read_start:], pretty=True)

### Route `/memories/` to a shared store
Read `/memories/preference.txt` using a different thread ID. This store does not contain the first example's `/notes/decision.txt`. The fixed namespace shares files for one fictional user; change it to isolate users. Sharing across threads does not provide disk persistence.


In [ ]:
# Execute
shared_store = InMemoryStore()
store_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=demo_namespace)},
)
store_agent = create_agent(
    model=get_model(), store=shared_store, checkpointer=InMemorySaver(),
    middleware=[FilesystemMiddleware(backend=store_backend), TraceMiddleware("11_store_files", pretty=True)],
)
store_writer_config = run_config("writer")
store_reader_config = run_config("reader")
store_written = store_agent.invoke({"messages": [("user", prompt("memory_write"))]}, config=store_writer_config)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(store_written["messages"], pretty=True)

In [ ]:
# Execute
store_read = store_agent.invoke({"messages": [("user", prompt("memory_read"))]}, config=store_reader_config)

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(store_read["messages"], pretty=True)

### List, search, edit and delete virtual files

`ls`, `glob` and `grep` locate files by directory, path pattern or content. `read_file` supports bounded line reads. `write_file` creates a file, `edit_file` replaces matching content, and `delete` removes it. A `tools` allowlist can expose only read/search operations; custom descriptions and a system-prompt addition guide tool use. Large tool results can be evicted to the backend instead of filling the conversation.

In [ ]:
# Execute
file_tools_agent = create_agent(model=ScriptedModel(responses=[
    tool_request("write_file", file_path="/scratch/example.txt", content="alpha\nbeta\n"),
    tool_request("ls", path="/scratch/"),
    tool_request("glob", pattern="**/*.txt"),
    tool_request("grep", pattern="beta", path="/scratch/"),
    tool_request("read_file", file_path="/scratch/example.txt", offset=1, limit=1),
    tool_request("edit_file", file_path="/scratch/example.txt", old_string="beta", new_string="gamma"),
    tool_request("read_file", file_path="/scratch/example.txt"),
    tool_request("delete", file_path="/scratch/example.txt"),
    AIMessage(content="The scratch file was created, searched, edited and removed."),
]), middleware=[FilesystemMiddleware(backend=StateBackend(), system_prompt="Use virtual paths under /scratch/.", custom_tool_descriptions={"ls": "List virtual files in the requested directory."})])
file_tools_result = file_tools_agent.invoke({"messages": [("user", "Exercise the virtual file operations.")]}, config=run_config())
read_only_filesystem = FilesystemMiddleware(backend=StateBackend(), tools=["ls", "read_file", "glob", "grep"])

In [ ]:
# Inspect output
show_messages(file_tools_result["messages"], pretty=True)
print("Remaining scratch files:", file_tools_result.get("files"))
assert all(m.status == "success" for m in file_tools_result["messages"] if m.type == "tool")
assert "/scratch/example.txt" not in file_tools_result.get("files", {})
assert {tool.name for tool in read_only_filesystem.tools} == {"ls", "read_file", "glob", "grep"}
assert "/notes/decision.txt" not in state_agent.get_state(run_config("other-notes-thread")).values.get("files", {})